In [2]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

# Параметры
# SYMBOL = "BTCUSDT"  # Пара для анализа
#SYMBOLS = [#'BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'USDCUSDT', 'SOLUSDT', 'TRXUSDT', 'DOGEUSDT', 'ADAUSDT','HYPEUSDT',
           #'BCHUSDT', 'LEOUSDT', 'XMRUSDT', 'LINKUSDT', 'USDEUSDT', 'CCUSDT', 'DAIUSDT', 'XLMUSDT', 'USD1USDT', 'LTCUSDT',
           #'AVAXUSDT', 'HBARUSDT', 'PYUSDT', 'SUIUSDT', 'ZECUSDT', 'SHIBUSDT', 'TONUSDT', 'CROUSDT', 'XAUTUSDT', 'WLFIUSDT',
           #'PAXGUSDT', 'DOTUSDT', 'UNIUSDT', 'MNTUSDT', 'PIUSDT', 'TAOUSDT', 'OKBUSDT', 'SKYUSDT', 'MUSDT', 'ASTERUSDT',
           #'USDGUSDT', 'AAVEUSDT', 'NEARUSDT', 'RLUSDUSDT', 'BGBUSDT', 'ICPUSDT', 'PEPEUSDT', 'ETCUSDT', 'ONDOUSDT', 'USDDUSDT',
           #'KSCUSDT', 'WLDUSDT', 'POLUSDT', 'UUSDT', 'ATOMUSDT', 'ENAUSDT', 'KASUSDT', 'GTUSDT', 'NIGHTUSDT', 'RENDERUSDT',
           #'MORPHOUSDT', 'ALGOUSDT', 'FLRUSDT', 'QNTUSDT', 'APTUSDT', 'TRUMPUSDT', 'FILUSDT', 'PUMPUSDT', 'ZROUSDT', 'XDCUSDT',
           #'VETUSDT', 'ARBUSDT', 'STABLEUSDT', 'NEXOUSDT', 'JUPUSDT', 'BONKUSDT', 'TUSDUSDT', 'DCRUSDT', 'KITEUSDT', 'JSTUSDT',
           #'STXUSDT', 'VIRTUALUSDT', 'CAKEUSDT', 'EURCUSDT', 'PENGUUSDT', 'SEIUSDT', 'ETHFIUSDT', 'DASHUSDT', 'CHZUSDT', 'XTZUSDT',
           #'FDUSDUSDT', 'FETUSDT', 'PIPPINUSDT', 'DEXEUSDT', 'CRVUSDT', 'KAIAUSDT', 'GNOUSDT', 'HUSDT', 'NFTUSDT'
           #]

SYMBOLS = [{'symbol': 'BTCUSDT'}]

INTERVAL = "1h"      # Дневные свечи (1d, 4h, 1h и т. д.)
LIMIT = 1000         # Максимальное количество свечей за запрос (максимум 1000 для Binance)

# Базовый URL API Binance
BASE_URL = "https://api.binance.com/api/v3/klines"

def get_top_binance_by_market_cap():
    # 1. получаем топ по market cap
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 250,
        "page": 1
    }
    coins = []

    for page in range(1, 5):  # берём 4 страницы (1000 монет)
        params = {
            "vs_currency": "usd",
            "order": "market_cap_desc",
            "per_page": 250,
            "page": page
        }
        response = requests.get(url, params=params).json()
        coins.extend(response)
        del coins[-1]

    # 2. получаем пары Binance
    binance_data = requests.get("https://api.binance.com/api/v3/exchangeInfo").json()
    symbols = [s['symbol'] for s in binance_data['symbols']]

    # 3. фильтруем только те, что есть на Binance
    result = []
    for coin in coins:
        symbol = coin['symbol'].upper() + "USDT"
        if symbol in symbols:
            result.append({
                "symbol": symbol,
                "market_cap": coin['market_cap']
            })

    return sorted(result, key=lambda x: x['market_cap'], reverse=True)[:200]

def get_binance_data(symbol, interval, limit):
    """
    Получает исторические данные с Binance через API.
    Возвращает DataFrame с данными.
    """
    # Рассчитываем timestamp для даты год назад
    end_time = int(datetime.now().timestamp() * 1000)  # Текущее время в мс
    start_time = int((datetime.now() - timedelta(days=3650)).timestamp() * 1000)  # Год назад в мс

    all_data = []
    current_start = start_time

    print("Начинаем загрузку данных...")

    while current_start < end_time:
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': current_start,
            'endTime': end_time,
            'limit': limit
        }

        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status()  # Проверяем статус ответа

            data = response.json()

            if not data:
                break  # Если данных нет, выходим

            # Добавляем полученные данные в общий список
            all_data.extend(data)

            # Обновляем start_time для следующего запроса (последняя свеча + 1 интервал)
            last_candle_time = data[-1][0]  # Время последней свечи в мс
            current_start = last_candle_time + 1

            print(f"Загружено {len(data)} свечей. Всего: {len(all_data)}")

            # Задержка, чтобы не превысить лимит запросов API
            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"Ошибка при запросе к API: {e}")
            break

    return all_data

def process_data(raw_data):
    """
    Обрабатывает сырые данные из API в удобный DataFrame.
    """
    if not raw_data:
        return pd.DataFrame()

    # Определяем столбцы согласно документации Binance API
    columns = [
        'timestamp', 'open', 'high', 'low', 'close',
        'volume', 'close_time', 'quote_asset_volume',
        'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume', 'ignore'
    ]

    df = pd.DataFrame(raw_data, columns=columns)

    # Преобразуем временные метки в читаемый формат
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')

    # Приводим числовые столбцы к нужному типу
    numeric_columns = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_columns] = df[numeric_columns].astype(float)

    return df

# Основной блок выполнения
if __name__ == "__main__":
    # Получаем данные
    #SYMBOLS = get_top_binance_by_market_cap()
    count = 0
    for elem in SYMBOLS:
        SYMBOL = elem['symbol']
        raw_data = get_binance_data(SYMBOL, INTERVAL, LIMIT)

        if raw_data:
            # Обрабатываем данные
            df = process_data(raw_data)
            count += 1

            # Сохраняем в CSV
            filename = f"data/{INTERVAL}_10_years/{SYMBOL}_{INTERVAL}_10_years.csv"
            df.to_csv(filename, index=False)
            print(f"\nДанные сохранены в файл: {filename}")

            # Выводим информацию о данных
            print(f"\nИнформация о данных:")
            print(f"Период: с {df['timestamp'].min()} по {df['timestamp'].max()}")
            print(f"Количество свечей: {len(df)}")
            print(f"Диапазон цен: от {df['low'].min():.2f} до {df['high'].max():.2f}")
        else:
            print("Не удалось получить данные.")
    print(f'Данных получено {count}')


Начинаем загрузку данных...
Загружено 1000 свечей. Всего: 1000
Загружено 1000 свечей. Всего: 2000
Загружено 1000 свечей. Всего: 3000
Загружено 1000 свечей. Всего: 4000
Загружено 1000 свечей. Всего: 5000
Загружено 1000 свечей. Всего: 6000
Загружено 1000 свечей. Всего: 7000
Загружено 1000 свечей. Всего: 8000
Загружено 1000 свечей. Всего: 9000
Загружено 1000 свечей. Всего: 10000
Загружено 1000 свечей. Всего: 11000
Загружено 1000 свечей. Всего: 12000
Загружено 1000 свечей. Всего: 13000
Загружено 1000 свечей. Всего: 14000
Загружено 1000 свечей. Всего: 15000
Загружено 1000 свечей. Всего: 16000
Загружено 1000 свечей. Всего: 17000
Загружено 1000 свечей. Всего: 18000
Загружено 1000 свечей. Всего: 19000
Загружено 1000 свечей. Всего: 20000
Загружено 1000 свечей. Всего: 21000
Загружено 1000 свечей. Всего: 22000
Загружено 1000 свечей. Всего: 23000
Загружено 1000 свечей. Всего: 24000
Загружено 1000 свечей. Всего: 25000
Загружено 1000 свечей. Всего: 26000
Загружено 1000 свечей. Всего: 27000
Загружено